In [177]:
%load_ext autoreload
%autoreload 2

import yaml

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

# epoch_path = "../../experiment_data/balance_metrics/cross_epoch.csv"
epoch_better_path = "../../experiment_data/balance_metrics/cross_epoch_better.csv"

# epoch_data = get_data(epoch_path)
epoch_data_better = pd.read_csv(epoch_better_path)

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

symbol_seq = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'star']
symbol_seq_train = ['circle-open', 'square-open', 'diamond-open', 'cross-open', 'x-open', 'triangle-up-open', 'triangle-down-open', 'star-open']
symbol_seq_val = ['circle-dot', 'square-dot', 'diamond-dot', 'cross-dot', 'x-dot', 'triangle-up-dot', 'triangle-down-dot', 'star-dot']

epoch_data_better["model"] = epoch_data_better["model"].apply(lambda x: f"{x}_better")
epoch_data_better.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,dataset,split,model,k,epoch,layer,thresh,thresh_mode,node_count,average_branching_factor,...,average_colless_index,total_colless,missing_colless,missing_colless_frac,sackin_index,average_sackin_index,leaf_count,total_volume,train_acc,val_acc
0,cifar,train,densenet_better,20,0,a1,0.0,0.0,2294,2.099817,...,5356.109658,994,98,0.089744,52616904.0,43774.462562,1202,50000,0.09992,0.0998
1,cifar,train,densenet_better,20,125,a1,0.0,0.0,710,2.055072,...,653.790123,324,21,0.060870,2991251.0,8195.208219,365,50000,0.99836,0.8659
2,cifar,train,densenet_better,20,150,a1,0.0,0.0,868,2.069212,...,680.676923,390,29,0.069212,2190354.0,4878.293987,449,50000,0.99990,0.8752
3,cifar,train,densenet_better,20,176,a1,0.0,0.0,306,2.046980,...,851.496454,141,8,0.053691,3013258.0,19192.726115,157,50000,0.97020,0.8464
4,cifar,train,densenet_better,20,200,a1,0.0,0.0,396,2.046632,...,960.269231,182,11,0.056995,2684267.0,13222.990148,203,50000,0.97902,0.8529


In [178]:
window_sizes = [3, 5, 7, 9, 11, 13]

In [179]:
epoch_trainUval = epoch_data_better[(epoch_data_better['split'] == 'trainUval')]
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig = make_subplots(specs=[[{"secondary_y": True}]], )

train_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='train_acc', color="dataset", color_discrete_sequence=color_seq_train, symbol="model")
val_acc_trainUval = px.line(epoch_trainUval, x='epoch', y='val_acc', color="dataset", color_discrete_sequence=color_seq_val, symbol="model", line_dash_sequence=['dash'])

traces = tuple(train_acc_trainUval.data)

for i, trace in enumerate(traces):
	# get dataset, model out of the trace
	dataset, model = str(trace['name']).split(", ")
    
	all_epochs = epoch_trainUval[(epoch_trainUval['dataset'] == dataset) & (epoch_trainUval['model'] == model)].copy()

	for size in window_sizes:     
		col_name = f"sackin_index_std_w{size}"
		all_epochs[col_name] = all_epochs["sackin_index"].rolling(size).std()
  
		group = f"{trace["name"]} {size}"
		sackin_scatter_trainUval = go.Scatter(x=all_epochs['epoch'], y=all_epochs[col_name], mode='markers', name=group, legend="legend2", legendgroup=group)

		fig.add_trace(
			sackin_scatter_trainUval,
			secondary_y=False
		)
  
		epoch_span = (all_epochs['epoch'].min(), all_epochs['epoch'].max())
		lowest_5_variances = all_epochs.nsmallest(10, col_name)[col_name].to_numpy()
		for j, e in enumerate(lowest_5_variances[[0, -1]]):
			xline = go.Scatter(x=epoch_span, y=[e, e], mode="lines", line=dict(dash=f"10px 10px"), legend="legend2", legendgroup=group, showlegend=False)
			fig.add_trace(xline, secondary_y=False)

 
	fig.update_yaxes(type="log", secondary_y=False)

	fig.add_trace(
		train_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	fig.add_trace(
		val_acc_trainUval.data[i],
		secondary_y=True,
	)
 
	top_5_epochs = all_epochs.sort_values(by='val_acc', ascending=False).iloc[:5]['epoch'].to_numpy()

	for i, e in enumerate(top_5_epochs):
		fig.add_trace(go.Scatter(x=[e, e], y=[0, 1.05], mode="lines", line=dict(color=trace['marker']['color'], dash=f"5px {i*5}px"), legendgroup=trace['name'], showlegend=False), secondary_y=True)

# set title
fig.update_layout(
	title_text="Cross-Epoch on TrainUVal: sackin Index Variance vs Train/Val Accuracy",
	legend = dict(title="Accuracy", tracegroupgap=0, x=1.05, y=1),
	legend2 = dict(title="sackin Index Variance", tracegroupgap=0, x=1.25, y=1),
)
fig.show()